In [2]:
from ultralytics import YOLO
import random
import cv2
import numpy as np
import time
import pytesseract

In [25]:
class LineDrawer:
    def __init__(self, img=None, img_path=None):
        if img is not None:
            self.img = img.copy()
        elif img_path is not None:
            self.img = cv2.imread(img_path)

        self.clone = self.img.copy()
        self.nameimg = "My Drawing"
        self.finished = False
        self.points = []
        self.lastpoints = []

    def draw_line(self, event, x, y, flags, param) :
        if event == cv2.EVENT_LBUTTONDOWN :
            self.points.append((x, y))
            cv2.circle(self.img, (x, y), 4, (0, 0, 255), -1)

            if len(self.points) == 4 :
                pts = np.array(self.points, np.int32).reshape((-1, 1, 2))
                cv2.polylines(self.img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
                self.finished = True
                self.lastpoints = self.points.copy()
                self.points.clear()

    def get_points(self) :
        if self.finished :
            self.finished = False
            return np.array(self.lastpoints, dtype=np.int32)
        return None
    
    def clear_draw(self) :
        self.finished = False
        self.points.clear()
        self.img = self.clone.copy()

    def setup(self):
        cv2.namedWindow(self.nameimg)
        cv2.setMouseCallback(self.nameimg, self.draw_line)
        while not self.finished:
            cv2.imshow(self.nameimg, self.img)
            key = cv2.waitKey(1) & 0xFF

            if key == ord('r'):
                self.clear_draw()

            elif key == ord('q'):
                break
        cv2.destroyAllWindows()


if __name__=="__main__":
    cap = cv2.VideoCapture("Lane.mp4")
    ret, frame = cap.read()
    if not ret:
        print("Error !!")

    ######## SET UP ########
    drawer = LineDrawer(img = frame)
    drawer.setup()
    Model_Car = YOLO("yolo11n.pt")

    ######## APP ########
    pts = drawer.get_points()
    frame_count = 0
    results = None

    while pts is not None:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        mask_poly = np.zeros(frame.shape[:2], dtype=np.uint8)

        if frame_count % 30 == 0:
            frame_count = 0
            results = Model_Car(frame, device=0, conf=0.35, classes=[2], verbose=False)
            boxes = results[0].boxes.xyxy.cpu().numpy()

            cv2.fillPoly(mask_poly, [pts], 255)
            intersection_view = np.zeros_like(frame)

            for (x1, y1, x2, y2) in boxes.astype(int):
                mask_car = np.zeros(frame.shape[:2], dtype=np.uint8)
                cv2.rectangle(mask_car, (x1, y1), (x2, y2), 255, -1)
                cv2.rectangle(frame, (x1, y1), (x2, y2), 255, -1)
                mask_inter = cv2.bitwise_and(mask_poly, mask_car)

                # คำนวณอัตราส่วนการทับ
                car_area = (x2 - x1) * (y2 - y1)
                inter_area = cv2.countNonZero(mask_inter)
                ratio = inter_area / car_area if car_area > 0 else 0

                if ratio >= 0.8:
                    intersection_view[mask_inter > 0] = (255, 255, 255)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"{ratio:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    
        cv2.imshow("Intersection Area", intersection_view)
        cv2.imshow(drawer.nameimg, frame)

        ######### Key Operate ##########
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break
    cv2.destroyAllWindows()

In [13]:
img = cv2.imread("runs/detect/predict/Test2.jpg")
pts = np.array([[10,150],[150,100],[300,150],[350,100],[310,20],[35,10]])
print(img.shape[:2])
## (1) Crop the bounding rect
#rect = cv2.boundingRect(pts)
#x,y,w,h = rect
#croped = img[y:y+h, x:x+w].copy()

## (2) make mask
#pts = pts - pts.min(axis=0)

#mask = np.zeros(croped.shape[:2], np.uint8)
#cv2.drawContours(mask, [pts], -1, (255, 255, 255), -1, cv2.LINE_AA)
#cv2.imwrite("mask.png", mask)

(720, 1280)
